In [ ]:
# ========== FRESH RETRAINING ON REAL miCLIP DATA (FIXED) ==========
import torch.optim as optim
from torch.utils.data import TensorDataset

# ---------- 1. Build tensors ----------
if 'data' not in dir():
    raise RuntimeError("❌ 'data' DataFrame not found. Run the extraction cell first.")

mapping = {'A': 1, 'U': 2, 'C': 3, 'G': 4}
X = torch.tensor([[mapping.get(b, 0) for b in seq] for seq in data['sequence']], dtype=torch.long)
y = torch.tensor(data['label'].values, dtype=torch.float32)

# ---------- 2. Train/val/test split ----------
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=256, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=256)
test_loader  = DataLoader(TensorDataset(X_test, y_test), batch_size=256)

# ---------- 3. Create a fresh model ----------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BiophysicalTensorFusionModel().to(device)   # uses class from extraction cell
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
criterion = nn.BCEWithLogitsLoss()

# ---------- 4. Training loop ----------
best_auroc = 0.0
patience, no_improve = 15, 0
epochs = 100

for epoch in range(1, epochs + 1):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model.classify(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)
    train_loss /= len(train_loader.dataset)

    # Validation
    model.eval()
    val_loss = 0.0
    preds, labels = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model.classify(xb)
            loss = criterion(logits, yb)
            val_loss += loss.item() * xb.size(0)
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            labels.extend(yb.cpu().numpy())
    val_loss /= len(val_loader.dataset)
    val_auroc = roc_auc_score(labels, preds)
    print(f"Epoch {epoch:2d} | Train Loss {train_loss:.4f} | Val Loss {val_loss:.4f} | Val AUROC {val_auroc:.4f}")

    if val_auroc > best_auroc:
        best_auroc = val_auroc
        no_improve = 0
        torch.save(model.state_dict(), "best_real_model.pt")
    else:
        no_improve += 1
        if no_improve >= patience:
            print("⏹ Early stopping triggered.")
            break

# ---------- 5. Final test evaluation ----------
model.load_state_dict(torch.load("best_real_model.pt"))
model.eval()
test_preds, test_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        probs = torch.sigmoid(model.classify(xb)).cpu().numpy()
        test_preds.extend(probs)
        test_labels.extend(yb.numpy())

test_auroc = roc_auc_score(test_labels, test_preds)
test_pred_bin = (np.array(test_preds) > 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(test_labels, test_pred_bin).ravel()

print("\n=========== REAL‑DATA RETRAINED (GSE63753 miCLIP) ===========")
print(f"AUROC:       {test_auroc:.4f}")
print(f"F1 Score:    {f1_score(test_labels, test_pred_bin):.4f}")
print(f"Sensitivity: {tp/(tp+fn):.4f}")
print(f"Specificity: {tn/(tn+fp):.4f}")
print("=============================================================")

# Save final model for deployment
torch.save(model.state_dict(), "EpiRNA_Biophysical_Master.pt")
print("✅ Retrained model saved as 'EpiRNA_Biophysical_Master.pt'.")w

Epoch  1 | Train Loss 0.7234 | Val Loss 0.7012 | Val AUROC 0.5593
Epoch  2 | Train Loss 0.6939 | Val Loss 0.6884 | Val AUROC 0.5711
Epoch  3 | Train Loss 0.6872 | Val Loss 0.6857 | Val AUROC 0.5813
Epoch  4 | Train Loss 0.6850 | Val Loss 0.6842 | Val AUROC 0.5895
Epoch  5 | Train Loss 0.6835 | Val Loss 0.6829 | Val AUROC 0.5964
Epoch  6 | Train Loss 0.6822 | Val Loss 0.6816 | Val AUROC 0.6017
Epoch  7 | Train Loss 0.6808 | Val Loss 0.6805 | Val AUROC 0.6059
Epoch  8 | Train Loss 0.6796 | Val Loss 0.6793 | Val AUROC 0.6093
Epoch  9 | Train Loss 0.6783 | Val Loss 0.6782 | Val AUROC 0.6126
Epoch 10 | Train Loss 0.6771 | Val Loss 0.6771 | Val AUROC 0.6153
Epoch 11 | Train Loss 0.6760 | Val Loss 0.6761 | Val AUROC 0.6183
Epoch 12 | Train Loss 0.6748 | Val Loss 0.6750 | Val AUROC 0.6212
Epoch 13 | Train Loss 0.6737 | Val Loss 0.6740 | Val AUROC 0.6234
Epoch 14 | Train Loss 0.6726 | Val Loss 0.6729 | Val AUROC 0.6262
Epoch 15 | Train Loss 0.6716 | Val Loss 0.6719 | Val AUROC 0.6284
Epoch 16 |

#maxxng out

## Complete transformer model code


In [ ]:
# ==========================================
# ALL-IN-ONE: Extract real data + Train Transformer
# ==========================================
!pip install -q transformers twobitreader

import gzip, os, random, re, torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
from transformers import AutoModel, AutoTokenizer
import torch.optim as optim
import twobitreader

# ----------------- 1. EXTRACT REAL miCLIP DATA -----------------
BED_FILE = "GSE63753_hek293.sysy.CITS.m6A.12051.bed.gz"
if not os.path.exists(BED_FILE):
    raise RuntimeError("❌ Upload the GEO BED file first (GSE63753_hek293.sysy.CITS.m6A.12051.bed.gz)")

# Load m6A sites
sites = []
with gzip.open(BED_FILE, 'rt') as f:
    for line in f:
        if line.startswith('#') or not line.strip():
            continue
        parts = line.strip().split()
        if len(parts) < 2: continue
        chrom, pos = parts[0], int(parts[1])
        sites.append((chrom, pos + 1))   # 1‑based coordinate
sites = list(set(sites))
print(f"Unique m6A sites: {len(sites)}")

# Load hg38 2-bit genome (download if missing)
genome_path = "/content/hg38.2bit"
if not os.path.exists(genome_path):
    !wget -q https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.2bit -O hg38.2bit
genome = twobitreader.TwoBitFile(genome_path)

# Extract 41‑nt positive windows
window = 20
pos_seqs = []
for chrom, one_based in sites:
    if chrom not in genome: continue
    seq_rec = genome[chrom]
    start = one_based - window - 1
    end   = one_based + window
    if start >= 0 and end <= len(seq_rec):
        snippet = seq_rec[start:end].upper().replace('T', 'U')
        if len(snippet) == 41 and all(b in 'AUCG' for b in snippet):
            pos_seqs.append(snippet)
print(f"Extracted {len(pos_seqs)} positive sequences")

# Create negative set (DRACH, non‑m6A)
drach_re = re.compile(r'[AGU][AG]AC[ACU]')
known_positions = set((chrom, one_based) for chrom, one_based in sites)
neg_seqs = []
attempts, max_attempts = 0, len(pos_seqs)*5
chrom_list = list(genome.keys())
while len(neg_seqs) < len(pos_seqs) and attempts < max_attempts:
    chrom = random.choice(chrom_list)
    seq_rec = genome[chrom]
    rpos = random.randint(window, len(seq_rec)-window-1)
    if (chrom, rpos+1) in known_positions:
        attempts += 1; continue
    snippet = seq_rec[rpos-window:rpos+window+1].upper().replace('T', 'U')
    if len(snippet) == 41 and drach_re.search(snippet):
        neg_seqs.append(snippet)
    attempts += 1
print(f"Generated {len(neg_seqs)} negative sequences")

n = min(len(pos_seqs), len(neg_seqs))
pos_final = random.sample(pos_seqs, n)
neg_final = random.sample(neg_seqs, n)
data = pd.DataFrame({'sequence': pos_final+neg_final, 'label': [1]*n + [0]*n})
data = data.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Final dataset: {n} pos, {n} neg → {len(data)} total")

# ----------------- 2. CROSS-SCALE GATE (reused from earlier) -----------------
class CrossScaleFusionGate(nn.Module):
    def __init__(self, channels=32):
        super().__init__()
        self.query_conv = nn.Conv1d(channels, channels//4, 1)
        self.key_conv   = nn.Conv1d(channels, channels//4, 1)
        self.value_conv = nn.Conv1d(channels, channels, 1)
        self.gamma      = nn.Parameter(torch.zeros(1))
    def forward(self, source, target):
        B, C, L = source.shape
        Q = self.query_conv(source).view(B, -1, L).permute(0,2,1)
        K = self.key_conv(target).view(B, -1, L)
        attn = F.softmax(torch.bmm(Q, K), dim=-1)
        V = self.value_conv(target).view(B, -1, L)
        out = torch.bmm(V, attn.permute(0,2,1)).view(B, C, L)
        return source + self.gamma * out

# ----------------- 3. TRANSFORMER-BIOPHYSICAL FUSION MODEL -----------------
tokenizer = AutoTokenizer.from_pretrained("armheb/DNA_bert_6")

def seq2kmer(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

class TransformerRNADataset(Dataset):
    def __init__(self, df):
        self.seqs = df['sequence'].values
        self.lbls = df['label'].values.astype(np.float32)
        self.mapping = {'A':1, 'U':2, 'C':3, 'G':4}
    def __len__(self): return len(self.lbls)
    def __getitem__(self, idx):
        seq = self.seqs[idx]
        kmer_seq = seq2kmer(seq)
        enc = tokenizer(kmer_seq, return_tensors="pt", padding="max_length", max_length=128, truncation=True)
        raw_tokens = torch.tensor([self.mapping.get(b,0) for b in seq], dtype=torch.long)
        return (enc['input_ids'].squeeze(0), enc['attention_mask'].squeeze(0),
                raw_tokens, torch.tensor(self.lbls[idx]))

class TransformerBiophysicalFusion(nn.Module):
    def __init__(self, freeze_bert=True):
        super().__init__()
        self.bert = AutoModel.from_pretrained("armheb/DNA_bert_6")
        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False
        # Biophysical embedding
        biophysical_matrix = torch.tensor([
            [0.0,  0.0,  0.0], [1.0, -1.0,  0.5], [-1.0, -1.0, -0.5],
            [-1.0,  1.0,  2.5], [1.0,  1.0, -1.0]
        ])
        self.bio_embed = nn.Embedding.from_pretrained(biophysical_matrix, freeze=False)
        self.local_path  = nn.Conv1d(3, 32, 3, padding=1)
        self.flank_path  = nn.Conv1d(3, 32, 5, padding=4, dilation=2)
        self.struct_path = nn.Conv1d(3, 32, 5, padding=8, dilation=4)
        self.fuse_local = CrossScaleFusionGate(32)
        self.fuse_flank = CrossScaleFusionGate(32)
        self.bio_norm  = nn.LayerNorm(96)
        self.bert_project = nn.Linear(768, 96)
        self.classifier = nn.Linear(192, 1)

    def forward(self, input_ids, attention_mask, raw_tokens):
        # Transformer
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_emb = bert_out.last_hidden_state[:, 0, :]          # (B,768)
        bert_feat = self.bert_project(cls_emb)                 # (B,96)
        # Biophysical CNN
        x_emb = self.bio_embed(raw_tokens).transpose(1,2)
        c1 = self.local_path(x_emb)
        c2 = self.flank_path(x_emb)
        c3 = self.struct_path(x_emb)
        c1 = self.fuse_local(c1, c3)
        c2 = self.fuse_flank(c2, c3)
        p1 = F.max_pool1d(F.pad(c1,(0,1)),2,1)
        p2 = F.max_pool1d(F.pad(c2,(0,1)),2,1)
        p3 = F.max_pool1d(F.pad(c3,(0,1)),2,1)
        combined = torch.cat([p1,p2,p3],1).transpose(1,2)
        bio_feat = F.relu(self.bio_norm(combined))
        bio_pooled = bio_feat.max(dim=1)[0]                    # (B,96)
        # Fusion
        fused = torch.cat([bio_pooled, bert_feat], dim=1)
        return self.classifier(fused).squeeze(-1)

# ----------------- 4. DATA LOADERS -----------------
full_dataset = TransformerRNADataset(data)
indices = list(range(len(full_dataset)))
train_idx, temp_idx = train_test_split(indices, test_size=0.3, random_state=42, stratify=data['label'])
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42, stratify=data.iloc[temp_idx]['label'])

def collate_fn(batch):
    input_ids, attention_mask, raw_tokens, labels = zip(*batch)
    return (torch.stack(input_ids), torch.stack(attention_mask),
            torch.stack(raw_tokens), torch.tensor(labels))

train_loader = DataLoader(torch.utils.data.Subset(full_dataset, train_idx), batch_size=128, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(torch.utils.data.Subset(full_dataset, val_idx), batch_size=256, collate_fn=collate_fn)
test_loader  = DataLoader(torch.utils.data.Subset(full_dataset, test_idx), batch_size=256, collate_fn=collate_fn)

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

# ----------------- 5. TRAINING -----------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TransformerBiophysicalFusion(freeze_bert=True).to(device)

optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50, T_mult=2)
criterion = nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

best_auroc, no_improve, patience = 0.0, 0, 30
max_epochs = 500

for epoch in range(1, max_epochs+1):
    model.train()
    train_loss = 0.0
    for input_ids, att_mask, raw_tok, labels in train_loader:
        input_ids, att_mask, raw_tok, labels = input_ids.to(device), att_mask.to(device), raw_tok.to(device), labels.to(device)
        optimizer.zero_grad()
        if scaler:
            with torch.amp.autocast('cuda'):
                logits = model(input_ids, att_mask, raw_tok)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(input_ids, att_mask, raw_tok)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
        train_loss += loss.item() * len(labels)
    train_loss /= len(train_loader.dataset)
    scheduler.step()

    model.eval()
    val_loss = 0.0; preds, lbls = [], []
    with torch.no_grad():
        for input_ids, att_mask, raw_tok, labels in val_loader:
            input_ids, att_mask, raw_tok, labels = input_ids.to(device), att_mask.to(device), raw_tok.to(device), labels.to(device)
            logits = model(input_ids, att_mask, raw_tok)
            loss = criterion(logits, labels)
            val_loss += loss.item() * len(labels)
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            lbls.extend(labels.cpu().numpy())
    val_loss /= len(val_loader.dataset)
    val_auroc = roc_auc_score(lbls, preds)
    print(f"Epoch {epoch:3d} | TrnLoss {train_loss:.4f} | ValLoss {val_loss:.4f} | ValAUROC {val_auroc:.4f}")

    if val_auroc > best_auroc:
        best_auroc = val_auroc; no_improve = 0
        torch.save(model.state_dict(), "best_transformer_model.pt")
    else:
        no_improve += 1
        if no_improve >= patience:
            print("Early stopping."); break

# ----------------- 6. EVALUATION -----------------
model.load_state_dict(torch.load("best_transformer_model.pt"))
model.eval()
test_preds, test_lbls = [], []
with torch.no_grad():
    for input_ids, att_mask, raw_tok, labels in test_loader:
        input_ids, att_mask, raw_tok = input_ids.to(device), att_mask.to(device), raw_tok.to(device)
        logits = model(input_ids, att_mask, raw_tok)
        test_preds.extend(torch.sigmoid(logits).cpu().numpy())
        test_lbls.extend(labels.numpy())

test_auroc = roc_auc_score(test_lbls, test_preds)
test_pred_bin = (np.array(test_preds)>0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(test_lbls, test_pred_bin).ravel()

print("\n=========== TRANSFORMER REAL‑DATA RESULT ===========")
print(f"AUROC:       {test_auroc:.4f}")
print(f"F1 Score:    {f1_score(test_lbls, test_pred_bin):.4f}")
print(f"Sensitivity: {tp/(tp+fn):.4f}")
print(f"Specificity: {tn/(tn+fp):.4f}")
print("=====================================================")

torch.save(model.state_dict(), "EpiRNA_Transformer.pt")
print("✅ Transformer model saved as 'EpiRNA_Transformer.pt'.")

Unique m6A sites: 12051
Extracted 11844 positive sequences
Generated 11844 negative sequences
Final dataset: 11844 pos, 11844 neg → 23688 total
Train: 16581, Val: 3553, Test: 3554


pytorch_model.bin:   0%|          | 0.00/359M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: armheb/DNA_bert_6
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch   1 | TrnLoss 0.6991 | ValLoss 0.6770 | ValAUROC 0.6387
Epoch   2 | TrnLoss 0.6637 | ValLoss 0.6587 | ValAUROC 0.6752
Epoch   3 | TrnLoss 0.6428 | ValLoss 0.6380 | ValAUROC 0.6990
Epoch   4 | TrnLoss 0.6187 | ValLoss 0.6146 | ValAUROC 0.7369
Epoch   5 | TrnLoss 0.5822 | ValLoss 0.5630 | ValAUROC 0.7684
Epoch   6 | TrnLoss 0.5376 | ValLoss 0.5233 | ValAUROC 0.7828
Epoch   7 | TrnLoss 0.5141 | ValLoss 0.5061 | ValAUROC 0.7833
Epoch   8 | TrnLoss 0.4946 | ValLoss 0.5005 | ValAUROC 0.7866
Epoch   9 | TrnLoss 0.4896 | ValLoss 0.4928 | ValAUROC 0.7873
Epoch  10 | TrnLoss 0.4813 | ValLoss 0.4896 | ValAUROC 0.7894
Epoch  11 | TrnLoss 0.4790 | ValLoss 0.4849 | ValAUROC 0.7960
Epoch  12 | TrnLoss 0.4727 | ValLoss 0.4899 | ValAUROC 0.7935
Epoch  13 | TrnLoss 0.4695 | ValLoss 0.4838 | ValAUROC 0.7942
Epoch  14 | TrnLoss 0.4666 | ValLoss 0.4819 | ValAUROC 0.7949
Epoch  15 | TrnLoss 0.4642 | ValLoss 0.4804 | ValAUROC 0.7974
Epoch  16 | TrnLoss 0.4626 | ValLoss 0.4880 | ValAUROC 0.7951
Epoch  1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp /content/EpiRNA_Transformer.pt /content/drive/MyDrive/

Mounted at /content/drive


#issues happend so now colong the main eprna test to test here

In [ ]:
git clone https://huggingface.co/spaces/supzammy/epirna-test
cd epirna-test

SyntaxError: invalid syntax (2626817030.py, line 1)

#pushing P now, pushing and retraining

In [1]:
# ==========================================
# 1. Install dependencies & import
# ==========================================
!pip install -q transformers twobitreader

import gzip, os, random, re, torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
from transformers import AutoModel, AutoTokenizer
import torch.optim as optim
import twobitreader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==========================================
# 2. Download the m6A BED file (upload manually if needed)
# ==========================================
# You must have the file 'GSE63753_hek293.sysy.CITS.m6A.12051.bed.gz' in Colab.
# If not, run this cell to download it:
!wget -q https://ftp.ncbi.nlm.nih.gov/geo/series/GSE63nnn/GSE63753/suppl/GSE63753_hek293.sysy.CITS.m6A.12051.bed.gz

BED_FILE = "GSE63753_hek293.sysy.CITS.m6A.12051.bed.gz"
if not os.path.exists(BED_FILE):
    raise RuntimeError("❌ BED file missing. Upload or download first.")

# ==========================================
# 3. Load m6A sites (DNA coordinates)
# ==========================================
sites = []
with gzip.open(BED_FILE, 'rt') as f:
    for line in f:
        if line.startswith('#') or not line.strip():
            continue
        parts = line.strip().split()
        if len(parts) < 2: continue
        chrom, pos = parts[0], int(parts[1])
        sites.append((chrom, pos + 1))   # 1‑based coordinate

sites = list(set(sites))
print(f"Unique m6A sites: {len(sites)}")

# ==========================================
# 4. Load hg38 2‑bit genome
# ==========================================
genome_path = "/content/hg38.2bit"
if not os.path.exists(genome_path):
    !wget -q https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.2bit -O hg38.2bit
genome = twobitreader.TwoBitFile(genome_path)

# ==========================================
# 5. Extract 41‑nt windows – KEEP DNA (T, not U)
# ==========================================
window = 20
pos_seqs = []
for chrom, one_based in sites:
    if chrom not in genome: continue
    seq_rec = genome[chrom]
    start = one_based - window - 1
    end   = one_based + window
    if start >= 0 and end <= len(seq_rec):
        snippet = seq_rec[start:end].upper()           # KEEP T, do NOT replace with U
        if len(snippet) == 41 and all(b in 'ATCG' for b in snippet):
            pos_seqs.append(snippet)
print(f"Extracted {len(pos_seqs)} positive sequences")

# ==========================================
# 6. Create negative set (DRACH‑containing, non‑m6A)
# ==========================================
drach_re = re.compile(r'[AGT][AG]AC[ACT]')            # DNA version of DRACH
known_positions = set((chrom, one_based) for chrom, one_based in sites)
neg_seqs = []
attempts, max_attempts = 0, len(pos_seqs) * 10
chrom_list = list(genome.keys())

while len(neg_seqs) < len(pos_seqs) and attempts < max_attempts:
    chrom = random.choice(chrom_list)
    seq_rec = genome[chrom]
    chrom_len = len(seq_rec)
    if chrom_len <= window * 2:
        attempts += 1; continue
    rpos = random.randint(window, chrom_len - window - 1)
    one_based = rpos + 1
    if (chrom, one_based) in known_positions:
        attempts += 1; continue
    snippet = seq_rec[rpos-window:rpos+window+1].upper()
    if len(snippet) == 41 and drach_re.search(snippet):
        neg_seqs.append(snippet)
    attempts += 1

print(f"Generated {len(neg_seqs)} negative sequences")

# Balance
n = min(len(pos_seqs), len(neg_seqs))
pos_final = random.sample(pos_seqs, n)
neg_final = random.sample(neg_seqs, n)
data = pd.DataFrame({'sequence': pos_final+neg_final, 'label': [1]*n + [0]*n})
data = data.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Final dataset: {n} pos, {n} neg → {len(data)} total")

# ==========================================
# 7. Cross‑Scale Fusion Gate (reuse)
# ==========================================
class CrossScaleFusionGate(nn.Module):
    def __init__(self, channels=32):
        super().__init__()
        self.query_conv = nn.Conv1d(channels, channels//4, 1)
        self.key_conv   = nn.Conv1d(channels, channels//4, 1)
        self.value_conv = nn.Conv1d(channels, channels, 1)
        self.gamma      = nn.Parameter(torch.zeros(1))
    def forward(self, source, target):
        B, C, L = source.shape
        Q = self.query_conv(source).view(B, -1, L).permute(0,2,1)
        K = self.key_conv(target).view(B, -1, L)
        attn = F.softmax(torch.bmm(Q, K), dim=-1)
        V = self.value_conv(target).view(B, -1, L)
        out = torch.bmm(V, attn.permute(0,2,1)).view(B, C, L)
        return source + self.gamma * out

# ==========================================
# 8. Transformer‑Biophysical Model (unchanged)
# ==========================================
tokenizer = AutoTokenizer.from_pretrained("armheb/DNA_bert_6")

def seq2kmer(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

class TransformerRNADataset(Dataset):
    def __init__(self, df):
        self.seqs = df['sequence'].values
        self.lbls = df['label'].values.astype(np.float32)
        self.mapping = {'A':1, 'T':2, 'C':3, 'G':4}   # T maps to 2 (same as U)
    def __len__(self): return len(self.lbls)
    def __getitem__(self, idx):
        seq = self.seqs[idx]
        kmer_seq = seq2kmer(seq)
        enc = tokenizer(kmer_seq, return_tensors="pt", padding="max_length", max_length=128, truncation=True)
        raw_tokens = torch.tensor([self.mapping.get(b,0) for b in seq], dtype=torch.long)
        return (enc['input_ids'].squeeze(0), enc['attention_mask'].squeeze(0),
                raw_tokens, torch.tensor(self.lbls[idx]))

class TransformerBiophysicalFusion(nn.Module):
    def __init__(self, freeze_bert=True):
        super().__init__()
        self.bert = AutoModel.from_pretrained("armheb/DNA_bert_6")
        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False
        biophysical_matrix = torch.tensor([
            [0.0,  0.0,  0.0], [1.0, -1.0,  0.5], [-1.0, -1.0, -0.5],
            [-1.0,  1.0,  2.5], [1.0,  1.0, -1.0]
        ])
        self.bio_embed = nn.Embedding.from_pretrained(biophysical_matrix, freeze=False)
        self.local_path  = nn.Conv1d(3, 32, 3, padding=1)
        self.flank_path  = nn.Conv1d(3, 32, 5, padding=4, dilation=2)
        self.struct_path = nn.Conv1d(3, 32, 5, padding=8, dilation=4)
        self.fuse_local = CrossScaleFusionGate(32)
        self.fuse_flank = CrossScaleFusionGate(32)
        self.bio_norm  = nn.LayerNorm(96)
        self.bert_project = nn.Linear(768, 96)
        self.classifier = nn.Linear(192, 1)

    def forward(self, input_ids, attention_mask, raw_tokens):
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_emb = bert_out.last_hidden_state[:, 0, :]
        bert_feat = self.bert_project(cls_emb)
        x_emb = self.bio_embed(raw_tokens).transpose(1,2)
        c1 = self.local_path(x_emb)
        c2 = self.flank_path(x_emb)
        c3 = self.struct_path(x_emb)
        c1 = self.fuse_local(c1, c3)
        c2 = self.fuse_flank(c2, c3)
        p1 = F.max_pool1d(F.pad(c1,(0,1)),2,1)
        p2 = F.max_pool1d(F.pad(c2,(0,1)),2,1)
        p3 = F.max_pool1d(F.pad(c3,(0,1)),2,1)
        combined = torch.cat([p1,p2,p3],1).transpose(1,2)
        bio_feat = F.relu(self.bio_norm(combined))
        bio_pooled = bio_feat.max(dim=1)[0]
        fused = torch.cat([bio_pooled, bert_feat], dim=1)
        return self.classifier(fused).squeeze(-1)

# ==========================================
# 9. DataLoaders
# ==========================================
full_dataset = TransformerRNADataset(data)
indices = list(range(len(full_dataset)))
train_idx, temp_idx = train_test_split(indices, test_size=0.3, random_state=42, stratify=data['label'])
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42, stratify=data.iloc[temp_idx]['label'])

def collate_fn(batch):
    input_ids, attention_mask, raw_tokens, labels = zip(*batch)
    return (torch.stack(input_ids), torch.stack(attention_mask),
            torch.stack(raw_tokens), torch.tensor(labels))

train_loader = DataLoader(torch.utils.data.Subset(full_dataset, train_idx), batch_size=128, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(torch.utils.data.Subset(full_dataset, val_idx), batch_size=256, collate_fn=collate_fn)
test_loader  = DataLoader(torch.utils.data.Subset(full_dataset, test_idx), batch_size=256, collate_fn=collate_fn)
print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

# ==========================================
# 10. Training
# ==========================================
model = TransformerBiophysicalFusion(freeze_bert=True).to(device)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50, T_mult=2)
criterion = nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

best_auroc, no_improve, patience = 0.0, 0, 30
max_epochs = 500

for epoch in range(1, max_epochs+1):
    model.train()
    train_loss = 0.0
    for input_ids, att_mask, raw_tok, labels in train_loader:
        input_ids, att_mask, raw_tok, labels = input_ids.to(device), att_mask.to(device), raw_tok.to(device), labels.to(device)
        optimizer.zero_grad()
        if scaler:
            with torch.amp.autocast('cuda'):
                logits = model(input_ids, att_mask, raw_tok)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(input_ids, att_mask, raw_tok)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
        train_loss += loss.item() * len(labels)
    train_loss /= len(train_loader.dataset)
    scheduler.step()

    model.eval()
    val_loss = 0.0; preds, lbls = [], []
    with torch.no_grad():
        for input_ids, att_mask, raw_tok, labels in val_loader:
            input_ids, att_mask, raw_tok, labels = input_ids.to(device), att_mask.to(device), raw_tok.to(device), labels.to(device)
            logits = model(input_ids, att_mask, raw_tok)
            loss = criterion(logits, labels)
            val_loss += loss.item() * len(labels)
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            lbls.extend(labels.cpu().numpy())
    val_loss /= len(val_loader.dataset)
    val_auroc = roc_auc_score(lbls, preds)
    print(f"Epoch {epoch:3d} | TrnLoss {train_loss:.4f} | ValLoss {val_loss:.4f} | ValAUROC {val_auroc:.4f}")

    if val_auroc > best_auroc:
        best_auroc = val_auroc; no_improve = 0
        torch.save(model.state_dict(), "best_transformer_model.pt")
    else:
        no_improve += 1
        if no_improve >= patience:
            print("Early stopping."); break

# ==========================================
# 11. Final test evaluation
# ==========================================
model.load_state_dict(torch.load("best_transformer_model.pt"))
model.eval()
test_preds, test_lbls = [], []
with torch.no_grad():
    for input_ids, att_mask, raw_tok, labels in test_loader:
        input_ids, att_mask, raw_tok = input_ids.to(device), att_mask.to(device), raw_tok.to(device)
        logits = model(input_ids, att_mask, raw_tok)
        test_preds.extend(torch.sigmoid(logits).cpu().numpy())
        test_lbls.extend(labels.numpy())

test_auroc = roc_auc_score(test_lbls, test_preds)
test_pred_bin = (np.array(test_preds)>0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(test_lbls, test_pred_bin).ravel()
print("\n=========== REAL‑DATA RESULT (DNA training) ===========")
print(f"AUROC:       {test_auroc:.4f}")
print(f"F1 Score:    {f1_score(test_lbls, test_pred_bin):.4f}")
print(f"Sensitivity: {tp/(tp+fn):.4f}")
print(f"Specificity: {tn/(tn+fp):.4f}")
print("======================================================")

torch.save(model.state_dict(), "EpiRNA_Transformer.pt")
print("✅ Retrained transformer model saved as 'EpiRNA_Transformer.pt'.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 82.0.1 which is incompatible.
Unique m6A sites: 12051
Extracted 11844 positive sequences
Generated 11844 negative sequences
Final dataset: 11844 pos, 11844 neg → 23688 total


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/28.7k [00:00<?, ?B/s]

...

Train: 16581, Val: 3553, Test: 3554


pytorch_model.bin:   0%|          | 0.00/359M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: armheb/DNA_bert_6
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch   1 | TrnLoss 0.6483 | ValLoss 0.6193 | ValAUROC 0.6942
Epoch   2 | TrnLoss 0.6041 | ValLoss 0.5749 | ValAUROC 0.7551
Epoch   3 | TrnLoss 0.5339 | ValLoss 0.5215 | ValAUROC 0.7782
Epoch   4 | TrnLoss 0.5078 | ValLoss 0.5107 | ValAUROC 0.7768
Epoch   5 | TrnLoss 0.5003 | ValLoss 0.5046 | ValAUROC 0.7820
Epoch   6 | TrnLoss 0.4946 | ValLoss 0.4938 | ValAUROC 0.7943
Epoch   7 | TrnLoss 0.4893 | ValLoss 0.4891 | ValAUROC 0.7964
Epoch   8 | TrnLoss 0.4800 | ValLoss 0.4823 | ValAUROC 0.8008
Epoch   9 | TrnLoss 0.4745 | ValLoss 0.4780 | ValAUROC 0.8055
Epoch  10 | TrnLoss 0.4720 | ValLoss 0.4744 | ValAUROC 0.8068
Epoch  11 | TrnLoss 0.4680 | ValLoss 0.4750 | ValAUROC 0.8025
Epoch  12 | TrnLoss 0.4661 | ValLoss 0.4722 | ValAUROC 0.8072
Epoch  13 | TrnLoss 0.4622 | ValLoss 0.4729 | ValAUROC 0.8045
Epoch  14 | TrnLoss 0.4620 | ValLoss 0.4695 | ValAUROC 0.8069
Epoch  15 | TrnLoss 0.4608 | ValLoss 0.4696 | ValAUROC 0.8077
Epoch  16 | TrnLoss 0.4589 | ValLoss 0.4731 | ValAUROC 0.8048
Epoch  1

#now download

In [2]:
from google.colab import files
files.download('EpiRNA_Transformer.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>